# Trivijalni baseline za procjenu smjera pogleda

Bez treniranja i bez slika: globalni srednji vektor i srednji vektor po ispitaniku. Svi srednji vektori računaju se **isključivo na trening skupu**. Evaluacija se provodi na testnom skupu.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

BASE_PATH = Path("/home/fipulab/anastazija/TEyeD_preprocessed/128x128_stride10_q4_seg_pupil-iris-lid_2D")
OUTPUT_PATH = Path("/home/fipulab/anastazija/runs/trivial_baseline")
GAZE_COLS = ["x", "y", "z"]
META_COLS = ["filename", "subject_id", "video_id"]
INPUT_COLS = ["filename", "x", "y", "z", "subject_id", "video_id"]
PRED_COLS = ["subject_id", "video_id", "filename", "gt_x", "gt_y", "gt_z", "pred_x", "pred_y", "pred_z", "angle_deg"]
EPS = 1e-6
UNIT_ATOL = 1e-5


/home/fipulab/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/fipulab/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
def load_and_validate(path: Path, split_name: str) -> pd.DataFrame:
    # Učitavaju se samo potrebni stupci; stupci mask_valid_* i slike se ne koriste.
    df = pd.read_csv(path, usecols=INPUT_COLS, dtype={"subject_id": "string", "video_id": "string", "filename": "string"})
    original_count = len(df)
    for col in GAZE_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    vectors = df[GAZE_COLS].to_numpy(dtype=np.float64)
    # Nevaljane oznake, NaN, beskonačnosti i nulti vektori nisu prikladni za kutnu pogrešku.
    invalid = (~np.isfinite(vectors).all(axis=1)) | np.all(vectors == -1.0, axis=1)
    invalid |= np.linalg.norm(np.where(np.isfinite(vectors), vectors, 0.0), axis=1) < EPS
    if invalid.any():
        warnings.warn(f"{split_name}: izbačeno {int(invalid.sum())} redaka s NaN/inf, (-1,-1,-1) ili nultim gaze vektorom.", stacklevel=2)
        df = df.loc[~invalid].copy().reset_index(drop=True)

    if df.empty:
        raise ValueError(f"{split_name}: nema valjanih gaze vektora nakon čišćenja.")

    vectors = df[GAZE_COLS].to_numpy(dtype=np.float64)
    norms = np.linalg.norm(vectors, axis=1)
    not_unit = ~np.isclose(norms, 1.0, atol=UNIT_ATOL, rtol=0.0)
    if not_unit.any():
        warnings.warn(f"{split_name}: {int(not_unit.sum())} vektora nije jedinične duljine; normaliziram ih.", stacklevel=2)
    df.loc[:, GAZE_COLS] = vectors / norms[:, None]
    if df[META_COLS].isna().any().any():
        warnings.warn(f"{split_name}: postoje nedostajući metapodaci (filename/subject_id/video_id).", stacklevel=2)
    print(f"{split_name}: {len(df)} valjanih od {original_count} uzoraka | "
          f"ispitanika: {df['subject_id'].nunique()} | videa: {df['video_id'].nunique()}")
    return df

train_df = load_and_validate(BASE_PATH / "train" / "labels.csv", "TRAIN")
test_df = load_and_validate(BASE_PATH / "test" / "labels.csv", "TEST")


TRAIN: 339466 valjanih od 339466 uzoraka | ispitanika: 76 | videa: 264
TEST: 79964 valjanih od 79964 uzoraka | ispitanika: 11 | videa: 50


In [3]:
# Srednji vektori računaju se samo iz TRAIN podataka.
raw_global = train_df[GAZE_COLS].to_numpy(dtype=np.float64).mean(axis=0)
global_norm_before = float(np.linalg.norm(raw_global))
if global_norm_before < EPS:
    warnings.warn("Globalni srednji gaze vektor je blizu nule — baseline je nepouzdan.", stacklevel=1)
    # Deterministički rezervni smjer kada normalizacija nije numerički pouzdana.
    mean_vec_global = np.array([0.0, 0.0, 1.0], dtype=np.float64)
else:
    mean_vec_global = raw_global / global_norm_before

subject_means = {}
subject_fallbacks = []
for subject_id, group in train_df.groupby("subject_id", dropna=False):
    raw_mean = group[GAZE_COLS].to_numpy(dtype=np.float64).mean(axis=0)
    norm = float(np.linalg.norm(raw_mean))
    if norm < EPS:
        warnings.warn(f"TRAIN ispitanik {subject_id}: srednji vektor blizu nule; koristim globalni baseline.", stacklevel=1)
        subject_means[subject_id] = mean_vec_global.copy()
        subject_fallbacks.append(str(subject_id))
    else:
        subject_means[subject_id] = raw_mean / norm

train_subjects = set(train_df["subject_id"].dropna())
test_subjects = set(test_df["subject_id"].dropna())
train_only = train_subjects - test_subjects
if train_only:
    warnings.warn(f"{len(train_only)} TRAIN ispitanika nema u TEST skupu; nastavljam.", stacklevel=1)
print(f"Norma globalnog srednjeg vektora prije normalizacije: {global_norm_before:.8f}")
print(f"Globalni jedinični srednji vektor: {mean_vec_global}")


Norma globalnog srednjeg vektora prije normalizacije: 0.99020656
Globalni jedinični srednji vektor: [ 0.02711324 -0.11720042  0.9927381 ]


/tmp/ipykernel_386289/3748862346.py:27: UserWarning: 76 TRAIN ispitanika nema u TEST skupu; nastavljam.
  warnings.warn(f"{len(train_only)} TRAIN ispitanika nema u TEST skupu; nastavljam.", stacklevel=1)


In [4]:
def ensure_unit(vectors: np.ndarray, name: str) -> None:
    norms = np.linalg.norm(vectors, axis=1)
    if not np.all(np.isfinite(vectors)) or not np.allclose(norms, 1.0, atol=UNIT_ATOL, rtol=0.0):
        raise ValueError(f"{name}: predikcije ili stvarne oznake nisu jedinični vektori.")


def evaluate(pred: np.ndarray, df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    gt = df[GAZE_COLS].to_numpy(dtype=np.float64)
    ensure_unit(pred, "Pred")
    ensure_unit(gt, "GT")
    angles = np.degrees(np.arccos(np.clip(np.einsum("ij,ij->i", pred, gt), -1.0, 1.0)))
    out = pd.DataFrame({
        "subject_id": df["subject_id"].to_numpy(),
        "video_id": df["video_id"].to_numpy(),
        "filename": df["filename"].to_numpy(),
        "gt_x": gt[:, 0], "gt_y": gt[:, 1], "gt_z": gt[:, 2],
        "pred_x": pred[:, 0], "pred_y": pred[:, 1], "pred_z": pred[:, 2],
        "angle_deg": angles,
    })[PRED_COLS]
    per_subject = out.groupby("subject_id", dropna=False)["angle_deg"].mean()
    summary = {
        "mean_angle": float(np.mean(angles)),
        "std_angle": float(np.std(angles, ddof=1)) if len(angles) > 1 else None,
        "median_angle": float(np.median(angles)),
        "p95_angle": float(np.percentile(angles, 95)),
        "num_subjects": int(df["subject_id"].nunique()),
        "num_samples": int(len(df)),
        "per_subject_mean_angle": {str(k): float(v) for k, v in per_subject.items()},
    }
    if len(angles) == 1:
        warnings.warn("Samo jedan testni uzorak: std (ddof=1) nije definiran.", stacklevel=1)
    return out, summary

# Globalni baseline: isti vektor za svaki testni uzorak.
pred_global = np.repeat(mean_vec_global[None, :], len(test_df), axis=0)

# Per-subject baseline: isključivo sredine ispitanika prisutnih u TRAIN skupu.
missing_subject_mask = ~test_df["subject_id"].isin(train_subjects)
missing_count = int(missing_subject_mask.sum())
if missing_count:
    warnings.warn(f"Per-subject: {missing_count} TEST uzoraka nema ispitanika u TRAIN skupu; koristim globalni fallback.", stacklevel=1)
pred_subject = np.stack([subject_means.get(sid, mean_vec_global) for sid in test_df["subject_id"]])

global_predictions, global_summary = evaluate(pred_global, test_df)
subject_predictions, subject_summary = evaluate(pred_subject, test_df)
global_summary.update({
    "baseline_type": "global",
    "mean_vec_global_norm_before_normalization": global_norm_before,
    "mean_vec_global": mean_vec_global.tolist(),
    "num_train_samples": int(len(train_df)),
    "num_train_subjects": int(train_df["subject_id"].nunique()),
    "num_train_videos": int(train_df["video_id"].nunique()),
})
subject_summary.update({
    "baseline_type": "per_subject",
    "num_test_samples_with_global_fallback": missing_count,
    "num_test_subjects_with_global_fallback": int(test_df.loc[missing_subject_mask, "subject_id"].nunique()),
    "train_subjects_with_near_zero_mean": subject_fallbacks,
    "note": "Cheating baseline: koristi subject_id; stvarni model ga ne koristi. Sredine se računaju samo iz TRAIN skupa.",
})


/tmp/ipykernel_386289/1441359863.py:41: UserWarning: Per-subject: 79964 TEST uzoraka nema ispitanika u TRAIN skupu; koristim globalni fallback.
  warnings.warn(f"Per-subject: {missing_count} TEST uzoraka nema ispitanika u TRAIN skupu; koristim globalni fallback.", stacklevel=1)


In [5]:
# Dva odvojena CSV-a, s točno zadanim redoslijedom stupaca.
for baseline_type, predictions, summary in [
    ("global", global_predictions, global_summary),
    ("per_subject", subject_predictions, subject_summary),
]:
    folder = OUTPUT_PATH / baseline_type
    folder.mkdir(parents=True, exist_ok=True)
    assert list(predictions.columns) == PRED_COLS
    predictions.to_csv(folder / "test_predictions.csv", index=False)
    with (folder / "summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2, allow_nan=False)

summary_cols = ["baseline_type", "mean_angle", "std_angle", "median_angle", "p95_angle", "num_subjects", "num_samples"]
pd.DataFrame([global_summary, subject_summary])[summary_cols].to_csv(OUTPUT_PATH / "summary.csv", index=False)

print("\nREZULTATI (kutna pogreška u stupnjevima)")
for name, summary in [("Globalni", global_summary), ("Per-subject", subject_summary)]:
    std_text = f"{summary['std_angle']:.4f}" if summary["std_angle"] is not None else "nedefinirano"
    print(f"{name}: {summary['mean_angle']:.4f} ± {std_text} ° | "
          f"medijan: {summary['median_angle']:.4f} ° | p95: {summary['p95_angle']:.4f} °")
print("NAPOMENA: per-subject je 'cheating' baseline jer koristi subject_id, koji stvarni model ne koristi.")
if missing_count == len(test_df):
    print("Svi TEST ispitanici su nepoznati u TRAIN skupu: per-subject predikcije jednake su globalnima.")
print(f"Rezultati spremljeni u: {OUTPUT_PATH}")



REZULTATI (kutna pogreška u stupnjevima)
Globalni: 6.1764 ± 4.8060 ° | medijan: 5.0588 ° | p95: 16.0632 °
Per-subject: 6.1764 ± 4.8060 ° | medijan: 5.0588 ° | p95: 16.0632 °
NAPOMENA: per-subject je 'cheating' baseline jer koristi subject_id, koji stvarni model ne koristi.
Svi TEST ispitanici su nepoznati u TRAIN skupu: per-subject predikcije jednake su globalnima.
Rezultati spremljeni u: /home/fipulab/anastazija/runs/trivial_baseline
